# Prompt Formatting and Structure Tutorial

## Overview

This tutorial explores various prompt formats and structural elements in prompt engineering, demonstrating their impact on AI model responses. We'll use an open-source model (Qwen3-14B) running locally via HuggingFace Transformers to experiment with different prompt structures and analyze their effectiveness.

## Motivation

Understanding how to format and structure prompts is crucial for effective communication with AI models. Well-structured prompts can significantly improve the quality, relevance, and consistency of AI-generated responses. This tutorial aims to provide practical insights into crafting prompts that elicit desired outcomes across various use cases.

## Key Components

1. Different prompt formats (Q&A, dialogue, instructions)
2. Structural elements (headings, bullet points, numbered lists)
3. Modern delimiter best practices (XML tags, markdown headers, triple backticks)
4. Comparison of prompt effectiveness
5. Best practices for prompt formatting

## Method Details

We'll use HuggingFace Transformers with a 4-bit quantized Qwen3-14B model on Google Colab. The tutorial will demonstrate:

1. Setting up the environment with HuggingFace Transformers and quantization
2. Creating various prompt formats (Q&A, dialogue, instructions)
3. Incorporating structural elements like headings, lists, and XML tags
4. Comparing responses from different prompt structures

Throughout the tutorial, we'll use a consistent theme (e.g., explaining a scientific concept) to showcase how different prompt formats and structures can yield varied results.

## Conclusion

By the end of this tutorial, you'll have a solid understanding of how prompt formatting and structure influence AI responses. You'll be equipped with practical techniques to craft more effective prompts, enhancing your ability to communicate with and leverage AI models for various applications.

## Setup

First, let's install HuggingFace Transformers, load a quantized open-source model, and define our helper function.

In [ ]:
# --- Google Colab Setup ---
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-14B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", quantization_config=quantization_config,
    cache_dir=CACHE_DIR, torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None, do_sample=do_sample, top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

def get_response(prompt):
    """Helper function to get model response and print it."""
    messages = [{"role": "user", "content": prompt}]
    response = generate(messages)
    print(response)
    print("-" * 50)
    return response

## Why Delimiters Help: Attention Boundary Detection

Before we explore different prompt formats, it's essential to understand *why* delimiters matter at a fundamental level.

### How the Transformer Sees Your Prompt

When you send a prompt to a language model, it doesn't see "sections" or "paragraphs" — it sees a flat sequence of tokens. The model's **self-attention mechanism** computes relationships between every pair of tokens in this sequence. Without any structural cues, every token attends to every other token with roughly equal opportunity, making it difficult for the model to distinguish between:

- **Instructions** ("Summarize the following article")
- **Context/data** (the actual article text)
- **Constraints** ("Keep it under 3 sentences")
- **Examples** (few-shot demonstrations)

Delimiters like XML tags (`<context>...</context>`), markdown headers (`###`), or triple backticks (` ``` `) create **attention boundaries** — tokens that signal "a new semantic region starts here." The model's attention weights spike at these boundary tokens, effectively creating soft partitions in the token sequence.

### The Attention Boundary Mechanism

1. **Boundary tokens have distinctive embeddings** — tags like `<task>` map to embeddings that are very different from natural language, creating a sharp contrast in the embedding space.
2. **Attention heads learn to gate on boundaries** — during training, the model learns that content between matching delimiters is semantically grouped. Attention heads develop patterns where they attend strongly within a delimited block and weakly across block boundaries.
3. **Position matters less, structure matters more** — with clear delimiters, the model can locate "the instructions" regardless of whether they appear at the beginning or middle of a long prompt.

Let's see this in action by comparing a delimited prompt with an undelimited one.

In [ ]:
# Compare: same task WITH and WITHOUT XML delimiters

article = """A recent study found that urban green spaces reduce ambient temperature
by 2-4°C during summer heatwaves. Researchers measured 50 parks across 12 cities
and found that parks larger than 2 hectares had the strongest cooling effect.
However, small pocket parks under 0.5 hectares showed negligible temperature
reduction, suggesting a minimum size threshold for meaningful urban cooling."""

# --- WITHOUT delimiters: instructions and data blend together ---
undelimited_prompt = f"""Summarize the following research finding in exactly
2 bullet points focusing only on the quantitative results. {article}"""

print("=== WITHOUT DELIMITERS ===")
resp_no_delim = get_response(undelimited_prompt)
print()

# --- WITH XML delimiters: clear boundaries between sections ---
delimited_prompt = f"""<task>Summarize the research finding in exactly 2 bullet points</task>

<context>
{article}
</context>

<constraints>
- Focus only on quantitative results (numbers, measurements)
- Exactly 2 bullet points, no more
</constraints>"""

print("=== WITH XML DELIMITERS ===")
resp_delim = get_response(delimited_prompt)
print()
print("Notice: The delimited version typically produces more focused, constraint-adherent output.")
print("The model can clearly separate WHAT to do (task) from WHAT to use (context) from HOW to do it (constraints).")

### Why XML Tags Are Especially Effective

XML/HTML tags are among the most effective delimiters for a specific reason: **training data composition**.

Large language models are trained on enormous corpora that include vast quantities of HTML and XML documents — web pages, configuration files, documentation, API responses, and structured data formats. As a result:

- The `<tag>` ... `</tag>` pattern is **deeply embedded** in the model's learned attention patterns
- The model has learned that content between matching tags is **semantically grouped**
- Opening and closing tags create a natural **bracket structure** that attention heads can exploit
- Tag names themselves carry semantic meaning — `<instruction>` signals "this is an instruction" at both the token level and the semantic level

This is why `<context>...</context>` works better than arbitrary delimiters like `***CONTEXT***` — the model has seen millions of examples of XML-style grouping during pre-training and has internalized the pattern deeply.

> **Key insight**: Effective prompt formatting isn't about what looks nice to humans — it's about leveraging patterns the model has already learned during training. XML tags, markdown headers, and code fences all work well because they appear frequently in training data as structural markers.

## Tokenization of Format Characters

Not all formatting characters are equal in the model's eyes. The way a delimiter **tokenizes** — whether it becomes a single token or gets split into multiple tokens — directly affects how "meaningful" it is as a structural boundary.

### Single-token vs Multi-token Delimiters

- A delimiter that tokenizes as a **single token** was learned as an atomic unit during training. The model treats it as one meaningful symbol with a single, strong embedding.
- A delimiter that splits into **multiple tokens** is reconstructed from parts. The model must "compose" the meaning across tokens, which can dilute its structural signal.

Let's inspect how common formatting elements tokenize with our model's tokenizer.

In [ ]:
# Examine how different formatting elements tokenize
format_elements = {
    "Markdown H1 (# )": "# ",
    "Markdown H3 (### )": "### ",
    "XML opening tag (<instruction>)": "<instruction>",
    "XML closing tag (</instruction>)": "</instruction>",
    "XML short tag (<task>)": "<task>",
    "XML short closing (</task>)": "</task>",
    "Triple backticks (```)": "```",
    "Horizontal rule (---)": "---",
    "Bullet point (- )": "- ",
    "Numbered list (1. )": "1. ",
    "Pipe separator (|)": "|",
    "Arrow (->)": "->",
    "Double newline": "\n\n",
    "Section marker (===)": "===",
}

print("Format Element".ljust(40) + "Token Count".rjust(11) + "  " + "Token IDs".rjust(20) + "  Decoded Tokens")
print("=" * 110)

for name, text in format_elements.items():
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    decoded = [tokenizer.decode([tid]) for tid in token_ids]
    ids_str = str(token_ids)
    dec_str = " | ".join(repr(d) for d in decoded)
    print(f"{name:<40} {len(token_ids):>11}  {ids_str:>20}  {dec_str}")

print()
print("KEY TAKEAWAY:")
print("Delimiters that tokenize as 1-2 tokens (like ### or <task>) are 'atomic' to the model.")
print("They carry strong, consistent structural signals learned during pre-training.")
print("Multi-token delimiters are reconstructed from parts and may be less reliable as boundaries.")

### Interpreting the Tokenization Results

What to look for in the output above:

- **Single-token delimiters** (like `###`, `---`, ` ``` `) — these are "first-class citizens" in the model's vocabulary. They were common enough in training data that the tokenizer learned to represent them as atomic units.
- **Multi-token delimiters** (like `<instruction>`) — these get split into pieces (`<`, `inst`, `ruction`, `>`). The model must compose their meaning across multiple positions, but because XML patterns are so common in training, the model still handles them well.
- **Common vs. rare patterns** — `<task>` (short, common in web data) may tokenize more efficiently than `<special_custom_delimiter>` (long, never seen in training).

> **Practical rule**: Prefer short, common delimiters. `<task>` beats `<my_custom_task_delimiter>`. `###` beats `=====SECTION=====`. The fewer tokens your delimiter consumes, the less of your context window it wastes, and the stronger its boundary signal.

## Exploring Different Prompt Formats

Let's explore various prompt formats using the topic of photosynthesis as our consistent theme.

### 1. Question and Answer (Q&A) Format

In [ ]:
qa_prompt = """Q: What is photosynthesis?
A:"""

get_response(qa_prompt)

### 2. Dialogue Format

In [ ]:
dialogue_prompt = """Student: Can you explain photosynthesis to me?
Teacher: Certainly! Photosynthesis is...
Student: What does a plant need for photosynthesis?
Teacher:"""

get_response(dialogue_prompt)

### 3. Instruction Format

In [ ]:
instruction_prompt = """Provide a brief explanation of photosynthesis, including its main components and importance."""

get_response(instruction_prompt)

## Impact of Structural Elements

Now, let's examine how structural elements like headings, lists, and modern delimiters affect the model's response.

### 1. Using Markdown Headings

In [ ]:
headings_prompt = """Explain photosynthesis using the following structure:

# Definition

# Process

# Importance
"""

get_response(headings_prompt)

### 2. Using Bullet Points

In [ ]:
bullet_points_prompt = """List the key components needed for photosynthesis:

• 
• 
• 
"""

get_response(bullet_points_prompt)

### 3. Using Numbered Lists

In [ ]:
numbered_list_prompt = """Describe the steps of photosynthesis in order:

1.
2.
3.
4.
"""

get_response(numbered_list_prompt)

### 4. Using XML Tags (Modern Best Practice)

XML tags provide clear section separation and are a state-of-the-art technique for structuring prompts. They help models parse distinct sections unambiguously.

In [ ]:
# XML-tag structured prompt (modern best practice)
article_text = """A 2024 study published in Nature found that global photosynthetic capacity has
increased by 12% over the past two decades, largely driven by rising CO2 levels and
longer growing seasons. However, researchers warn that heat stress and drought could
reverse these gains, potentially reducing crop yields by up to 25% in tropical regions
by 2050."""

xml_prompt = f"""<task>Summarize the following article about photosynthesis research</task>
<context>
{article_text}
</context>
<constraints>
- Maximum 3 sentences
- Include key statistics
- Use formal tone
</constraints>"""

get_response(xml_prompt)

### 5. Using Triple Backticks for Code/Data Sections

Triple backticks clearly delineate code or data blocks within prompts, preventing the model from confusing data with instructions.

In [ ]:
backtick_prompt = """Analyze the following experimental data on photosynthesis rates and identify the trend:

```data
Light Intensity (lux) | O2 Production (mL/min)
100                   | 0.5
500                   | 2.1
1000                  | 3.8
2000                  | 5.2
5000                  | 5.5
10000                 | 5.6
```

Provide your analysis in a short paragraph."""

get_response(backtick_prompt)

### 6. Combined Modern Formatting (XML + Markdown + Numbered Steps)

Combining multiple structural elements yields the most precise control over model output.

In [ ]:
combined_prompt = """<task>Explain the process of photosynthesis for a high-school biology student</task>

<instructions>
## Steps to follow
1. Start with a one-sentence definition
2. List the inputs and outputs as bullet points
3. Describe the two main stages (light-dependent and Calvin cycle)
4. End with a real-world analogy
</instructions>

<constraints>
- Keep the total response under 200 words
- Use simple language (no jargon without explanation)
- Format your answer with markdown headings for each step
</constraints>"""

get_response(combined_prompt)

## Format Robustness Testing

How much does the *format* of a prompt affect the *quality* of the response? Let's run a controlled experiment: same task, five different formatting approaches, three generations each. This tests whether the model produces consistent, high-quality output regardless of how the prompt is structured.

We'll use a moderately complex task — classifying a paragraph's sentiment and extracting key themes — to see where formatting makes a real difference.

In [ ]:
# Format Robustness Testing: Same task, 5 formats, 3 runs each
import textwrap

review_text = """The new city library is architecturally stunning with its floor-to-ceiling
windows and open atrium design. However, the acoustics are terrible — conversations on the
ground floor echo up to the reading rooms on the third floor. The digital catalog system
is intuitive and fast, but the physical book collection has shrunk by 40% compared to the
old building. Staff are friendly but clearly understaffed during peak hours."""

# Five different formatting approaches for the same task
format_prompts = {
    "Plain text": f"""Analyze the sentiment of this review and list the top 3 themes.
Review: {review_text}
Provide sentiment (positive/negative/mixed) and themes as a short list.""",

    "Markdown headers": f"""# Task
Analyze the sentiment and extract key themes from the review below.

# Review
{review_text}

# Output Format
- Sentiment: positive, negative, or mixed
- Top 3 themes as a bullet list""",

    "XML tags": f"""<task>Analyze sentiment and extract key themes</task>
<input>{review_text}</input>
<output_format>
- Sentiment: positive / negative / mixed
- Top 3 themes as bullet points
</output_format>""",

    "JSON-style": f"""{{
  "task": "Analyze sentiment and extract key themes",
  "input": "{review_text}",
  "output_format": {{
    "sentiment": "positive | negative | mixed",
    "themes": ["theme1", "theme2", "theme3"]
  }}
}}""",

    "Numbered sections": f"""[1] TASK: Analyze the sentiment and extract key themes from the review.
[2] INPUT: {review_text}
[3] REQUIREMENTS:
    a) Classify sentiment as positive, negative, or mixed
    b) List the top 3 themes""",
}

# Run each format 3 times to test consistency
for fmt_name, prompt in format_prompts.items():
    print(f"\n{'=' * 60}")
    print(f"FORMAT: {fmt_name}")
    print(f"{'=' * 60}")
    for run in range(1, 4):
        print(f"\n--- Run {run} ---")
        get_response(prompt)

### Interpreting the Robustness Results

When comparing across the five formats, look for:

1. **Consistency across runs** — Does the model give roughly the same answer each time? High variance suggests the format isn't providing strong enough structural cues.
2. **Format compliance** — Does the model follow the requested output format (bullet points, sentiment label)? Some formats make the expected output structure clearer than others.
3. **Content quality** — Does the model identify the same themes regardless of formatting?

**Typical findings with Qwen3 and similar models:**

- **XML tags** and **markdown headers** tend to produce the most consistent results — they create the clearest attention boundaries.
- **Plain text** works surprisingly well for simple tasks but breaks down as task complexity increases.
- **JSON-style** prompts can confuse the model into thinking you want a JSON *response* rather than just using JSON as a structural format.
- **Numbered sections** are robust but slightly less effective than XML for separating instructions from data.

> **Takeaway**: For critical applications, test your specific format with your specific model. Don't assume what works for one model works for all. XML and markdown are safe defaults for most modern models.

## Structured Output via Format Instructions

One of the most practical applications of prompt formatting is **requesting structured output**. When you need the model to respond in a specific format (JSON, markdown table, numbered list), the way you *show* the desired format matters far more than how you *describe* it.

### Show, Don't Tell

Language models are fundamentally **pattern completion engines**. When you provide an example of the desired output format, the model's next-token prediction naturally continues that pattern. This is far more reliable than verbally describing a format.

Let's test three approaches: describing the format, showing a template, and providing a complete example.

In [ ]:
# Structured Output: Comparing "describe" vs "template" vs "full example"

task = "List three benefits of renewable energy with a confidence score for each."

# Approach 1: Describe the format in words
describe_prompt = f"""{task}

Respond in JSON format with an array of objects, each having a "benefit" field
and a "confidence" field with a value between 0 and 1."""

print("=== APPROACH 1: Describe the format ===")
get_response(describe_prompt)

# Approach 2: Provide a template to fill in
template_prompt = f"""{task}

Use this exact format:
```json
[
  {{"benefit": "...", "confidence": 0.X}},
  {{"benefit": "...", "confidence": 0.X}},
  {{"benefit": "...", "confidence": 0.X}}
]
```"""

print("=== APPROACH 2: Provide a template ===")
get_response(template_prompt)

# Approach 3: Full example + task (most reliable)
example_prompt = f"""Here is an example of the output format:

```json
[
  {{"benefit": "Reduces air pollution from fossil fuel combustion", "confidence": 0.95}},
  {{"benefit": "Creates local manufacturing jobs", "confidence": 0.72}}
]
```

Now, {task.lower()}
Respond in the same JSON format shown above."""

print("=== APPROACH 3: Full example (most reliable) ===")
get_response(example_prompt)

print()
print("COMPARISON:")
print("- Approach 1 (describe): Model must interpret verbal format description → inconsistent")
print("- Approach 2 (template): Model sees the pattern skeleton → better compliance")
print("- Approach 3 (full example): Model sees a complete instance → best format compliance")

### Why Showing Examples Works So Well

This comes directly from how next-token prediction works:

1. **Pattern inertia** — When the model sees `[{"benefit": "...",` it has extremely high probability of continuing with `"confidence":` because the pattern is already established. The model's autoregressive nature means it strongly prefers to continue existing patterns.

2. **Reduced ambiguity** — "Respond in JSON" is ambiguous: should the keys be camelCase or snake_case? Should numbers be strings or actual numbers? An example resolves all these micro-decisions simultaneously.

3. **Implicit constraint propagation** — When your example shows exactly 2 items with specific field names and value types, the model learns the *complete schema* — field names, data types, array structure, and approximate value ranges — all from a single example.

> **Best practice**: For structured output, always include at least one complete example. The few extra tokens are a small price for dramatically improved format compliance. For production use, consider using the model's built-in JSON mode or a grammar-constrained decoder for guaranteed valid output.

## When Formatting Hurts

Formatting is a tool, not a universal improvement. There are specific cases where adding structure to a prompt actually *degrades* the model's output. Understanding these failure modes is just as important as knowing how to format well.

In [ ]:
# Anti-pattern 1: Over-formatted prompt (formatting noise overwhelms content)

over_formatted_prompt = """<system_context>
  <role>You are a helpful assistant</role>
  <expertise>General knowledge</expertise>
  <tone>Professional</tone>
</system_context>

<task_definition>
  <primary_objective>Answer the question below</primary_objective>
  <secondary_objective>Be concise</secondary_objective>
</task_definition>

<input_data>
  <question_type>Factual</question_type>
  <question>What is the capital of France?</question>
</input_data>

<output_specification>
  <format>Plain text</format>
  <max_length>One sentence</max_length>
  <style>Direct answer</style>
</output_specification>"""

# The same task, formatted simply
simple_prompt = """What is the capital of France? Answer in one sentence."""

print("=== OVER-FORMATTED (17 lines of XML for a trivial question) ===")
get_response(over_formatted_prompt)
print()
print("=== SIMPLE AND DIRECT ===")
get_response(simple_prompt)
print()
print("The over-formatted version wastes tokens on structure that adds zero informational")
print("value. The model must parse 17 lines of XML to find a 9-word question.")
print(f"Over-formatted tokens: {len(tokenizer.encode(over_formatted_prompt))}")
print(f"Simple tokens: {len(tokenizer.encode(simple_prompt))}")

In [ ]:
# Anti-pattern 2: Mismatched format signals (XML input but asking for JSON output)

mismatched_prompt = """<data>
  <item><name>Widget A</name><price>29.99</price><stock>150</stock></item>
  <item><name>Widget B</name><price>49.99</price><stock>30</stock></item>
  <item><name>Widget C</name><price>19.99</price><stock>200</stock></item>
</data>

Convert the above data to JSON format and add a "total_value" field
for each item (price × stock)."""

# Better: consistent format (JSON example in prompt for JSON output)
consistent_prompt = """Convert this product data to JSON and add a "total_value" field (price × stock).

Input data:
| Name     | Price  | Stock |
|----------|--------|-------|
| Widget A | 29.99  | 150   |
| Widget B | 49.99  | 30    |
| Widget C | 19.99  | 200   |

Example output for one item:
```json
{"name": "Widget A", "price": 29.99, "stock": 150, "total_value": 4498.50}
```

Now output all items as a JSON array."""

print("=== MISMATCHED: XML input → JSON output (format confusion) ===")
get_response(mismatched_prompt)
print()
print("=== CONSISTENT: Neutral input + JSON example → JSON output ===")
get_response(consistent_prompt)
print()
print("When the input format (XML) conflicts with the desired output format (JSON),")
print("the model sometimes continues the XML pattern instead of switching to JSON.")
print("Using a format-neutral input (like a markdown table) avoids this conflict.")

### Formatting Principles: When to Add and When to Subtract

Based on the experiments above, here are practical guidelines:

| Situation | Recommendation |
|-----------|---------------|
| Simple factual question | Minimal or no formatting — just ask clearly |
| Multi-step task with context | XML tags or markdown headers to separate sections |
| Requesting structured output | Provide a complete output example |
| Very long context (>2000 tokens) | Delimiters are critical — the model needs landmarks |
| Format conversion task | Use the *target* format in your prompt, not the source format |

**The formatting overhead rule**: If your format instructions and delimiters contain more tokens than your actual content, you're over-formatting. Formatting should be scaffolding, not the building itself.

> **Key insight**: Formatting is a spectrum. Too little and the model loses track of what's what. Too much and the model drowns in structural noise. The sweet spot depends on task complexity — match the amount of structure to the amount of ambiguity in your task.

## Comparing Prompt Effectiveness

Let's compare the effectiveness of different prompt structures for a specific task.

In [ ]:
comparison_prompts = [
    # Prompt 1: Minimal, unstructured
    "Explain the importance of photosynthesis for life on Earth.",

    # Prompt 2: Structured with numbered sections
    """Explain the importance of photosynthesis for life on Earth. Structure your answer as follows:
1. Oxygen production
2. Food chain support
3. Carbon dioxide absorption""",

    # Prompt 3: XML-structured with constraints (modern best practice)
    """<task>Explain why photosynthesis is important for life on Earth</task>
<format>
1. Oxygen production
2. Food chain support
3. Carbon dioxide absorption
</format>
<constraints>
- One paragraph per point
- Include at least one specific fact or number per point
</constraints>""",
]

for i, prompt in enumerate(comparison_prompts, 1):
    print(f"Prompt {i}:")
    get_response(prompt)